# Conservation score & nucleotide LFC  
PhastCons is sensitive to "runs" of conserved sites, and is therefore effective for picking out conserved elements. PhyloP, on the other hand, is more appropriate for evaluating signatures of selection at particular nucleotides or classes of nucleotides  
Downloaded phyloP values differ slightly from what are shown on the browser

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pyranges as pr
import re
import pingouin as pg
import statsmodels.api as sm
import statsmodels.formula.api as smf
import scipy.stats as stats
from Bio import SeqIO
from scipy.stats import ks_2samp
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

from matplotlib import font_manager
import matplotlib as mpl

arial_path = "/media/scratch/fy2306/tools/fonts"
font_files = font_manager.findSystemFonts(fontpaths=arial_path)

for file in font_files:
    font_manager.fontManager.addfont(file)
    
mpl.rcParams['font.family'] = 'Arial'
import warnings
warnings.filterwarnings("ignore")

In [ ]:
id_prefix_list = [
	"MYC_GFP_"
]

In [ ]:
# remove negative controls that can be mapped to the genome (with bowtie)
mapped_ctrl_path = "/media/scratch/fy2306/projects/base_editing/data/bowtie/all/all_sgrna_seqs.bowtie_hg38.processed.tsv"
mapped_ctrl_df = pd.read_csv(mapped_ctrl_path, sep="\t")
mapped_ctrl_df = mapped_ctrl_df[
    mapped_ctrl_df['id'].str.startswith(('Random', 'Non-targeting-controls', 'AAVS1'))
    & ~(mapped_ctrl_df['id'].str.startswith('AAVS1') & (mapped_ctrl_df['Alignments_NM0'] == 1) & (mapped_ctrl_df['Alignments_NM1'] == 0))
]
mapped_ctrl_list = mapped_ctrl_df["id"].tolist()
print(len(mapped_ctrl_list))
all_ctrl = [
    record.id for record in SeqIO.parse("/media/scratch/fy2306/projects/base_editing/data/bowtie/negative_controls/input/negative_controls.fa", "fasta")
    ]
clean_ctrl_list = list(set(all_ctrl)-set(mapped_ctrl_list))
print(len(clean_ctrl_list))
print(len(all_ctrl))

In [ ]:
# randomly group nc sgrnas
def group_and_aggregate(df, group_size, random_state=None):
    df_shuffled = df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    num_groups = len(df_shuffled) // group_size
    df_trimmed = df_shuffled.iloc[:num_groups * group_size]
    
    grouped = np.array_split(df_trimmed, num_groups)
    
    aggregated = []
    for group in grouped:
        sgrna_concat = ','.join(group['sgrna'].astype(str))
        avg_lfc = group['per time unit LFC'].mean()
        aggregated.append({
            'sgrna': sgrna_concat,
            'per time unit LFC': avg_lfc
        })
    
    return pd.DataFrame(aggregated)

In [ ]:
def conservation_lfc_cdf(
	gene, myc_var, cons_col,
	date,
	test_name,
	nc_test_name,
	merge_method_nucleotide, 
	win_size,
	baseline,
	treatments,
	day_numbers,
	cons_subset,
	merge_method_replicate, 
	conservation_method,
	conservation_threshold,
	title=None,
	per_time_unit=True, score_smoothing=False, abs_LFC=False):

	score_path = f"/media/scratch/fy2306/projects/base_editing/data/conservation/chr8_myc.{conservation_method}100way.wigFix"

	# preprocess
	df_path = f"/media/scratch/fy2306/projects/base_editing/data/{date}/mageck/{test_name}/{merge_method_nucleotide}/myc_repeats/{win_size}/MYC_U1.gene_summary.txt"
	df = pd.read_csv(df_path, sep="\t")
	pattern = f"^({'|'.join(map(re.escape, id_prefix_list))})([0-9]*\.?[0-9]+)"
	df = df[df['id'].str.match(pattern)]
	edited_pos_chr_col = "indel_base_pos_chr"	# nucleotide pos
	df[edited_pos_chr_col] = df['id'].str.extract(pattern)[1].astype(int)	# 1-based

	# merge
	treatment_lfc_columns = [f"{treatment}_vs_{baseline}|neg|lfc" for treatment in treatments]

	if per_time_unit:
		for col, day in zip(treatment_lfc_columns, day_numbers):
			df[col] = df[col] / (day + 0) # per time unit.

	if merge_method_replicate == "mean":
		df["neg|lfc"] = df[treatment_lfc_columns].mean(axis=1)

	if abs_LFC:
		df["neg|lfc"] = df["neg|lfc"].abs()

	# add repeats
	rmsk_path = "/media/dna/fy2306/genomes/hg38/RepeatMasker/rmsk.txt"
	rmsk_colnames = ["bin", "swScore", "milliDiv", "milliDel", "milliIns", "genoName", "genoStart", "genoEnd", "genoLeft", "strand", "repName", "repClass", "repFamily", "repStart", "repEnd", "repLeft", "id"]
	rmsk = pd.read_csv(rmsk_path, sep="\t", header=None, names=rmsk_colnames)
	rmsk = rmsk[rmsk['genoName'] == "chr8"]
	# rmsk: 0-based
	rmsk_ranges = pr.PyRanges(rmsk.rename(columns={
	'genoName': 'Chromosome',
	'genoStart': 'Start',
	'genoEnd': 'End'
	}))

	# base_pos: 1-based
	df['Start'] = df[edited_pos_chr_col] - 1
	df['End'] = df['Start'] + 1
	df['Chromosome'] = 'chr8'

	df["orig_index"] = df.index
	df_ranges = pr.PyRanges(df[['Chromosome', 'Start', 'End', 'orig_index']])

	overlap = df_ranges.intersect(rmsk_ranges, how="containment")	# in pyranges: “containment” reports intervals where the overlapping is contained within it, gives you the intervals in self be completely within the intervals in other
	overlap_indices = overlap.df['orig_index'].unique()

	df["indel_repeats"] = False
	df.loc[overlap_indices, "indel_repeats"] = True

	df = df[["id", edited_pos_chr_col, "neg|lfc", "indel_repeats"]]

	# process conservation
	conservation_scores = pd.read_csv(score_path, sep="\t", header=None, names=['chrom', edited_pos_chr_col, 'score'])
	# position: 1-based
	if score_smoothing:
		# rolling
		conservation_scores = conservation_scores.sort_values(by=edited_pos_chr_col).reset_index(drop=True)

		half_win = (int(win_size) // 2)

		smoothed_scores = []

		positions = conservation_scores[edited_pos_chr_col].values
		scores = conservation_scores['score'].values

		for i, pos in enumerate(positions):
			start = pos - half_win	# inclusive
			end = pos + half_win	# inclusive
			
			mask = (positions >= start) & (positions <= end)
			window_scores = scores[mask]
			
			smoothed_scores.append(window_scores.mean() if len(window_scores) > 0 else float('nan'))

		conservation_scores['score'] = smoothed_scores

	df = pd.merge(df, conservation_scores, on=edited_pos_chr_col, how="left")

	# load in lib
	lib_path = f"/media/scratch/fy2306/projects/base_editing/data/grna_type/MYC-lib-for-mageck.type.{myc_var}.organized.txt"
	df_lib = pd.read_csv(lib_path, sep="\t", usecols=[edited_pos_chr_col, cons_col])
	df_lib = df_lib.drop_duplicates(subset=edited_pos_chr_col)
	df = pd.merge(df, df_lib, on=edited_pos_chr_col, how='left')
	df = df[df["indel_repeats"] == False]
	# df: "id", edited_pos_chr_col, "neg|lfc", "indel_repeats", "chrom", "score", cons_col
	print(df[edited_pos_chr_col].duplicated().sum())
	# add promoter annotation
	df.loc[
		(df[edited_pos_chr_col] >= (127736084-1000)) & (df[cons_col] == 'upstream_flanking'),
		cons_col
		] = 'promoter'

	if cons_subset != "all":
		df = df[df[cons_col].isin(cons_subset)]
	if isinstance(conservation_threshold, float):
		conservation_threshold = conservation_threshold
	elif conservation_threshold == "mean":
		# mean of selected regions
		conservation_threshold = df['score'].mean()
	elif conservation_threshold == "median":
		# median of selected regions
		conservation_threshold = df['score'].median()

	bins = [float('-inf'), conservation_threshold, float('inf')]
	# labels = [f'(-∞, {conservation_threshold:.2f})', f'[{conservation_threshold:.2f}, ∞)']
	labels = ["Other", "Conserved"]

	df['score_bin'] = pd.cut(df['score'], bins=bins, labels=labels, right=False)
	print(df['score_bin'].value_counts())


	# process the negative controls
	df_nc = None
	for treatment in treatments:
		df_nc_path = f"/media/scratch/fy2306/projects/base_editing/data/{date}/mageck/{nc_test_name}/MYC_U1.{treatment}_vs_{baseline}.sgrna_summary.txt"
		df_nc_rep = pd.read_csv(df_nc_path, sep="\t", usecols=["sgrna", "Gene", "LFC"])
		df_nc_rep = df_nc_rep[df_nc_rep["Gene"].isin(["AAVS1", "NT", "Random"])]
		df_nc_rep = df_nc_rep.drop('Gene', axis=1)
		df_nc_rep = df_nc_rep.rename(columns={"LFC": f"LFC_{treatment}"})
		if df_nc is None:
			df_nc = df_nc_rep
		else:
			# Merge on 'sgrna'
			df_nc = pd.merge(df_nc, df_nc_rep, on="sgrna", how="inner")
	print(len(df_nc))

	treatment_lfc_nc_columns = [f"LFC_{treatment}" for treatment in treatments]

	if per_time_unit:
		for col, day in zip(treatment_lfc_nc_columns, day_numbers):
			df_nc[col] = df_nc[col] / (day + 0) # per time unit.

	if merge_method_replicate == "mean":
		df_nc["per time unit LFC"] = df_nc[treatment_lfc_nc_columns].mean(axis=1)

	# remove mismatches
	df_nc = df_nc[~(df_nc["sgrna"].isin(mapped_ctrl_list))]
	df_nc = df_nc[['sgrna', 'per time unit LFC']]

	# random group of df_nc
	n_repeats = 1000
	all_results = []

	for seed in range(n_repeats):
		df_nc_result = group_and_aggregate(df_nc, group_size=2*int(win_size), random_state=seed)
		all_results.append(df_nc_result)

	df_nc_combined = pd.concat(all_results, ignore_index=True)
	nc_combined_median_value = df_nc_combined["per time unit LFC"].median()
	print(nc_combined_median_value)

	df["neg|lfc"] = df["neg|lfc"] - nc_combined_median_value

	# cdf
	plt.figure(figsize=(4,4))

	colors = {
		"Other": "#66CCFE",
		"Conserved": "#FF0066"
	}

	for label in ["Conserved", "Other"]:
		group_data = df[df['score_bin'] == label]['neg|lfc'].sort_values()
		if len(group_data) == 0:
			continue
		cdf = np.arange(1, len(group_data)+1) / len(group_data)
		
		plt.plot(group_data, cdf, label=f'{conservation_method} {label}\n(N={len(group_data)})', color=colors[label])

	group_low = df[df['score_bin'] == "Other"]['neg|lfc']
	median_low = group_low.median()
	group_high = df[df['score_bin'] == "Conserved"]['neg|lfc']
	median_high = group_high.median()
	ks_stat, p_value = ks_2samp(group_low, group_high)

	shift = abs(median_high - median_low)

	if abs_LFC:
		plt.xlabel('Base-pair phenotype score (abs)', fontsize=14)
	else:
		plt.xlabel('Base-pair phenotype score', fontsize=14)
	plt.ylabel('CDF', fontsize=14)
	if title:
		plt.title(f"{title}", fontsize=13)
	else:
		plt.title(f"{cons_subset}\nscore_smoothing: {score_smoothing}, KS stat={ks_stat:.3f}, p={p_value:.2e}, shift={shift:.2e}", fontsize=13)
	mant, exp = f"{p_value:.2e}".split("e")
	mant = float(mant); exp = int(exp)
	handles = [
		Line2D([], [], linestyle="None", marker=None, color=colors["Conserved"],
			label=f'Conserved (N={len(df[df.score_bin=="Conserved"]):,})'),
		Line2D([], [], linestyle="None", marker=None, color=colors["Other"],
			label=f'Other (N={len(df[df.score_bin=="Other"]):,})'),
		Line2D([], [], linestyle="None", marker=None, color="black",
			label=rf"$P={mant:.2f}\times 10^{{{exp}}}$"),
		Line2D([], [], linestyle="None", marker=None, color="black",
			label=f'Threshold: {conservation_method} {conservation_threshold:.2f}'),
	]
	plt.legend(handles=handles, loc="upper left", frameon=False, fontsize=13,
			handlelength=0, handletextpad=0, labelcolor="linecolor")
	
	plt.yticks([0, 0.5, 1], fontsize=13)
	plt.xticks(fontsize=13)
	xmin = df[df["score_bin"].isin(labels)]["neg|lfc"].quantile(0.03)
	xmax = df[df["score_bin"].isin(labels)]["neg|lfc"].quantile(0.97)
	plt.xlim(xmin, xmax)
	plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(0.025))
	# plt.axvline(x = 0, color='grey', linestyle="--", alpha=0.5)
	# plt.axhline(y = 0.5, color='grey', linestyle="--", alpha=0.5)
	plt.gca().spines['top'].set_visible(False)
	plt.gca().spines['right'].set_visible(False)
	plt.grid(False)
	plt.savefig(f"/media/scratch/fy2306/projects/base_editing/plots/conservation/{conservation_method}_score_cdf.{title}.thresh{conservation_threshold}.pdf", 
			bbox_inches="tight",
			dpi=300,              
			transparent=True,
			format='pdf')
	plt.show()

In [ ]:
gene = "MYC"
myc_var = "myc2_indel"
cons_col = "indel_consequences_more"
date = "20250416"
test_name = "test-1-standard-nucleotide/test"
nc_test_name = "test-1-standard/test"
merge_method_nucleotide = "mean"
win_size = "4"
baseline = "Cas9-HMOI_D0"
treatments = ["Cas9-LMOI_D20", "Cas9-LMOI_D8", "Cas9_SpRY_L_MOI_D_20", "Cas9_SpRY_L_MOI_D_8"]
day_numbers = [20, 8, 20, 8]
cons_subset = ["upstream_flanking", "promoter", "5_prime_UTR", "intron_1", "intron_2", "intron_splice", "3_prime_UTR", "downstream_flanking"]
merge_method_replicate = "mean"
conservation_method = "phyloP"
conservation_threshold = "mean"
conservation_lfc_cdf(
	gene, myc_var, cons_col,
	date,
	test_name,
	nc_test_name,
	merge_method_nucleotide, 
	win_size,
	baseline,
	treatments,
	day_numbers,
	cons_subset,
	merge_method_replicate, 
	conservation_method, 
	conservation_threshold,
	title="Non-coding base-pairs",
	per_time_unit=True, score_smoothing=True, abs_LFC=False)